In [1]:
import utils.gcp_handling
import utils.components as components
import os
from datetime import datetime, timedelta
# Import necessary libraries
from kfp.v2 import dsl
from kfp.v2.dsl import component
from kfp.v2 import compiler
from google.cloud import aiplatform

c:\Program Files\Python\313\Lib\site-packages\kfp\dsl\component_decorator.py:126: FutureWarning: The default base_image used by the @dsl.component decorator will switch from 'python:3.9' to 'python:3.10' on Oct 1, 2025. To ensure your existing components work with versions of the KFP SDK released after that date, you should provide an explicit base_image argument and ensure your component works as intended on Python 3.10.
  return component_factory.create_component_from_func(


In [2]:
# Production constants - update these for each scoring run
user_constants = {
    "EMAIL": "sahil_aetna_com",
    "COSTCENTER": "13070",
    "TENANT": "hcm-cm-de",
    "USE_COMPUTE_PROJECT": True,
    "OWNER": "sahil_aetna_com",
    "COMPUTE_PROJECT": "anbc-dev-hcm-cm-de",
    "PROJECT": "anbc-dev-hcm-cm-de",
    'LABELS': {
        'owner': 'sahil_aetna_com',
        'costcenter': '13070',
        'tenant': 'hcm-cm-de',
        'self_serve': 'true',
        'lob': 'hcb',
        'pipeline_type': 'scoring'
    },
    "LOB": "hcb",
    "MODEL_DESCRIPTION": "maternity_training_data_pull",
    "PIPELINE_TYPE": "prediction",
    
    # SQL Variables for BigQuery queries
    "GCP_PROJECT": "anbc-hcb-dev",
    "GCP_DB": "cm_medicaid_hcb_dev",
    "PREFIX": "a534354_mlops_demo",
    "DEFAULT_EXP": "INTERVAL 3 DAY",
    
    # Production date variables - UPDATE THESE FOR EACH SCORING RUN
    "INDEX_DT": "CURRENT_DATE()",  # Use CURRENT_DATE() for real-time scoring
    "KMDO_DT": "'2025-01-15'",  # Komodo data date - update to latest available
    "SDOH_YR": "2024",  # SDOH year - use most recent available
    "DOB_ST": "'2022-01-01'",  # Baby DOB window start (training 001b)
    "DOB_END": "'2025-03-31'",  # Baby DOB window end (training 001b)
}

In [3]:
constants = user_constants
os.environ["OWNER"] = constants["EMAIL"]
os.environ["COSTCENTER"] = constants["COSTCENTER"]
os.environ["SDOH_YR"] = constants["SDOH_YR"]

In [4]:
def process_sql_file(sql_file_path, constants, base_path="../sql_queries/"):
    import os
    import re
    
    # Construct full path
    full_path = os.path.join(base_path, sql_file_path)
    
    # Read the SQL file content
    with open(full_path, 'r') as file:
        sql_content = file.read()
    
    # Find all variables in the SQL file
    variables_in_sql = re.findall(r'\{([^}]+)\}', sql_content)
    
    # Create a clean substitution dictionary from constants
    substitution_dict = {}
    
    for key, value in constants.items():
        # Skip nested dictionaries and non-string values that aren't useful for SQL
        if isinstance(value, dict):
            continue  # Skip LABELS and other nested objects
        elif isinstance(value, bool):
            substitution_dict[key] = str(value).upper()  # Convert True/False to TRUE/FALSE
        else:
            substitution_dict[key] = str(value)  # Convert everything to string
    
    # Add additional mappings for common SQL variable patterns
    additional_mappings = {
        'COST_CENTER': constants.get('COSTCENTER', ''),
    }
    
    # Merge additional mappings
    substitution_dict.update(additional_mappings)
    
    # Check which variables can be substituted
    missing_variables = []
    available_substitutions = {}
    
    for var_name in variables_in_sql:
        if var_name in substitution_dict:
            available_substitutions[var_name] = substitution_dict[var_name]
        else:
            missing_variables.append(var_name)
    
    # Warn about missing variables but don't fail
    if missing_variables:
        print(f"Warning: Variables not found in constants: {missing_variables}")
        print(f"Available substitutions: {list(substitution_dict.keys())}")
        print(f"Variables in SQL: {variables_in_sql}")
    
    # Substitute variables using **kwargs
    try:
        sql_query = sql_content.format(**available_substitutions)
    except KeyError as e:
        print(f"Error: Missing variable {e} in SQL file {sql_file_path}")
        print(f"Available substitutions: {list(available_substitutions.keys())}")
        raise
    
    return sql_query


In [5]:
from google.cloud import aiplatform
import os

# Define the production pipeline
@dsl.pipeline(
    name="maternity-training-data-pipeline",
    description="Training data pipeline for maternity model building"
)
def prod_bigquery_pipeline(
    project_id: str = "anbc-dev-hcm-cm-de",
    region: str = "us-east4",
    cmek_key: str = "projects/cvs-key-vault-nonprod/locations/us-east4/keyRings/gkr-nonprod-us-east4/cryptoKeys/gk-anbc-dev-hcm-cm-de-us-east4"
):
    """
    Training Data Pipeline Dependency Graph:
    
    PHASE 0: Cohort Definition
    ├── 001a_asdb_cpt_rev_icd (creates _all_icd, _all_procedure, _all_diagnoses)
    ├── 001b_asdb_cohort_id (after 001a)
    └── 001c_membership_cross_walk (after 001b)
    
    PHASE 1: Parallel execution (after 001c)
    ├── 001e_asdb_labs
    ├── 001f_edw_cpt_rev_icd
    ├── 001g_edw_labs
    ├── 001h_komodo_cpt_rev_icd
    ├── 002_Med_Claims
    ├── 004_Conditions
    ├── 006_Rx_Claims
    └── 007_Demographics

    PHASE 2: Secondary dependencies
    ├── 003_Cost_and_Utilization (after 002)
    ├── 010_preventative (after 003)
    ├── 008_GeoID (after 007)
    ├── 009_ACS (after 008)
    └── 011_CSDI_risk (after 008)
    
    PHASE 3: Aggregation
    ├── 001i_custom_predictors_joined (after 001a, 001e, 001f, 001h)
    ├── 001j_labs_joined (after 001e, 001g)
    └── 012_non_embedding_feature_beast (after 003, 004, 006, 007, 009, 010, 011)
    
    PHASE 4: Final Join
    └── 013_join_cust_pipeline_embed_features (after 001i, 001j, 012)
    """

    #################################
    # PHASE 0: Cohort Definition (TRAINING-SPECIFIC)
    # Unlike production which pulls from PREGNANCY_TABLE_W,
    # training builds cohort from historical birth records
    #################################
    
    # 001a: CPT/REV/ICD codes (creates _all_diagnoses needed by 001b)
    sql_001a = process_sql_file('001a_asdb_cpt_rev_icd.sql', constants)
    q_001a = components.query_bigquery_component(
        constants=constants,
        query=sql_001a,
        display_name="001a_asdb_cpt_rev_icd"
    )
    
    # 001b: Training Cohort ID (creates _final_timepoint from historical births)
    sql_001b = process_sql_file('001b_asdb_cohort_id.sql', constants)
    q_001b = components.query_bigquery_component(
        constants=constants,
        query=sql_001b,
        display_name="001b_asdb_cohort_id"
    ).after(q_001a)
    
    # 001c: Membership Cross Walk (creates _id_crosswalk for EDW/Komodo linking)
    sql_001c = process_sql_file('001c_membership_cross_walk.sql', constants)
    q_001c = components.query_bigquery_component(
        constants=constants,
        query=sql_001c,
        display_name="001c_membership_cross_walk"
    ).after(q_001b)

    #################################
    # PHASE 1: Parallel execution (after 001c)
    #################################

    # 001e: ASDB Labs
    sql_001e = process_sql_file('001e_asdb_labs.sql', constants)
    q_001e = components.query_bigquery_component(
        constants=constants,
        query=sql_001e,
        display_name="001e_asdb_labs"
    ).after(q_001c)

    # 001f: EDW CPT/REV codes
    sql_001f = process_sql_file('001f_edw_cpt_rev_icd.sql', constants)
    q_001f = components.query_bigquery_component(
        constants=constants,
        query=sql_001f,
        display_name="001f_edw_cpt_rev_icd"
    ).after(q_001c)

    # 001g: EDW Labs
    sql_001g = process_sql_file('001g_edw_labs.sql', constants)
    q_001g = components.query_bigquery_component(
        constants=constants,
        query=sql_001g,
        display_name="001g_edw_labs"
    ).after(q_001c)
    
    # 001h: Komodo CPT/REV/ICD
    sql_001h = process_sql_file('001h_komodo_cpt_rev_icd.sql', constants)
    q_001h = components.query_bigquery_component(
        constants=constants,
        query=sql_001h,
        display_name="001h_komodo_cpt_rev_icd"
    ).after(q_001c)

    # 002: Medical Claims (OPTIMIZED - single scan with year_flag)
    sql_002 = process_sql_file('002_Med_Claims.sql', constants)
    q_002 = components.query_bigquery_component(
        constants=constants,
        query=sql_002,
        display_name="002_Med_Claims"
    ).after(q_001c)

    # 004: Conditions
    sql_004 = process_sql_file('004_Conditions.sql', constants)
    q_004 = components.query_bigquery_component(
        constants=constants,
        query=sql_004,
        display_name="004_Conditions"
    ).after(q_001c)
   
    # 006: Rx Claims (OPTIMIZED - single scan with year_flag)
    sql_006 = process_sql_file('006_Rx_Claims.sql', constants)
    q_006 = components.query_bigquery_component(
        constants=constants,
        query=sql_006,
        display_name="006_Rx_Claims"
    ).after(q_001c)
    
    # 007: Demographics
    sql_007 = process_sql_file('007_Demographics.sql', constants)
    q_007 = components.query_bigquery_component(
        constants=constants,
        query=sql_007,
        display_name="007_Demographics"
    ).after(q_001c)
    
    #################################
    # PHASE 2: Secondary dependencies
    #################################

    # 003: Cost and Utilization (OPTIMIZED - creates all tables with year_flag)
    sql_003 = process_sql_file('003_Cost_and_Utilization.sql', constants)
    q_003 = components.query_bigquery_component(
        constants=constants,
        query=sql_003,
        display_name="003_Cost_and_Utilization"
    ).after(q_002)

    # 010: Preventative Care
    sql_010 = process_sql_file('010_preventative.sql', constants)
    q_010 = components.query_bigquery_component(
        constants=constants,
        query=sql_010,
        display_name="010_preventative"
    ).after(q_003)
   
    # 008: GeoID
    sql_008 = process_sql_file('008_GeoID.sql', constants)
    q_008 = components.query_bigquery_component(
        constants=constants,
        query=sql_008,
        display_name="008_GeoID"
    ).after(q_007)
     
    # 009: ACS Data
    sql_009 = process_sql_file('009_ACS.sql', constants)
    q_009 = components.query_bigquery_component(
        constants=constants,
        query=sql_009,
        display_name="009_ACS"
    ).after(q_008)

    # 011: CSDI Risk
    sql_011 = process_sql_file('011_CSDI_risk.sql', constants)
    q_011 = components.query_bigquery_component(
        constants=constants,
        query=sql_011,
        display_name="011_CSDI_risk"
    ).after(q_008)

    #################################
    # PHASE 3: Aggregation
    #################################
    
    # 001i: Custom Predictors Joined
    sql_001i = process_sql_file('001i_custom_predictors_joined.sql', constants)
    q_001i = components.query_bigquery_component(
        constants=constants,
        query=sql_001i,
        display_name="001i_custom_predictors_joined"
    ).after(*[q_001a, q_001e, q_001f, q_001h])
    
    # 001j: Labs Joined
    sql_001j = process_sql_file('001j_labs_joined.sql', constants)
    q_001j = components.query_bigquery_component(
        constants=constants,
        query=sql_001j,
        display_name="001j_labs_joined"
    ).after(*[q_001e, q_001g])
    
    # 012: Non-Embedding Feature Beast (joins pre-created summary tables from 003 and 006)
    sql_012 = process_sql_file('012_non_embedding_feature_beast.sql', constants)
    q_012 = components.query_bigquery_component(
        constants=constants,
        query=sql_012,
        display_name="012_non_embedding_feature_beast"
    ).after(*[q_003, q_004, q_006, q_007, q_009, q_010, q_011])

    #################################
    # PHASE 4: Final Join
    #################################
    
    sql_013 = process_sql_file('013_join_cust_pipeline_embed_features.sql', constants)
    q_013 = components.query_bigquery_component(
        constants=constants,
        query=sql_013,
        display_name="013_join_cust_pipeline_embed_features"
    ).after(*[q_001i, q_001j, q_012])

In [7]:
# Compile and run the production pipeline
if __name__ == "__main__":
    compiler.Compiler().compile(
        pipeline_func=prod_bigquery_pipeline,
        package_path="prod_bigquery_pipeline.json"
    )

    # Initialize Vertex AI client
    aiplatform.init(
        project="anbc-dev-hcm-cm-de",
        location="us-east4",
    )

    # Define the pipeline job
    pipeline_job = aiplatform.PipelineJob(
        display_name="maternity-prod-scoring-pipeline",
        template_path="prod_bigquery_pipeline.json",
        pipeline_root="gs://hcm-cm-de-code-hcb-dev/vertex-test/",
        encryption_spec_key_name="projects/cvs-key-vault-nonprod/locations/us-east4/keyRings/gkr-nonprod-us-east4/cryptoKeys/gk-anbc-dev-hcm-cm-de-us-east4",
        enable_caching=False,  # Disable caching for training to ensure fresh data pull
        parameter_values={
            "project_id": "anbc-dev-hcm-cm-de",
            "region": "us-east4",
            "cmek_key": "projects/cvs-key-vault-nonprod/locations/us-east4/keyRings/gkr-nonprod-us-east4/cryptoKeys/gk-anbc-dev-hcm-cm-de-us-east4"
        },
        labels={
            "owner": "palmere1_aetna_com",
            "pipeline_type": "scoring",
            "lob": "hcb",
            "costcenter": "13070",
            "tenant": "hcm-cm-de",
            "self_serve": "true"
        }
    )

    # Submit the pipeline job
    pipeline_job.run(
        service_account="gchcb-hcm-cm-de-ontpd@anbc-dev-hcm-cm-de.iam.gserviceaccount.com",
        sync=True
    )

Creating PipelineJob
PipelineJob created. Resource name: projects/46378383599/locations/us-east4/pipelineJobs/maternity-training-data-pipeline-20260325212105
To use this PipelineJob in another session:
pipeline_job = aiplatform.PipelineJob.get('projects/46378383599/locations/us-east4/pipelineJobs/maternity-training-data-pipeline-20260325212105')
View Pipeline Job:
https://console.cloud.google.com/vertex-ai/locations/us-east4/pipelines/runs/maternity-training-data-pipeline-20260325212105?project=46378383599
PipelineJob projects/46378383599/locations/us-east4/pipelineJobs/maternity-training-data-pipeline-20260325212105 current state:
3
PipelineJob projects/46378383599/locations/us-east4/pipelineJobs/maternity-training-data-pipeline-20260325212105 current state:
3
PipelineJob projects/46378383599/locations/us-east4/pipelineJobs/maternity-training-data-pipeline-20260325212105 current state:
3
PipelineJob projects/46378383599/locations/us-east4/pipelineJobs/maternity-training-data-pipeline-